# Results Dashboard

This notebook loads all saved experiment results and generates paper-ready tables and figures.
Run this after completing all experiments (notebooks 03-07) to produce publication materials.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "vit_obfuscation").exists():
            return path
    raise RuntimeError("Could not find repository root. Start Jupyter inside the repository.")


ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_DIR = ROOT / "configs" / "experiments"
CANONICAL_OUTPUT_DIR = ROOT / "outputs" / "revision_v3"
NOTEBOOK_OUTPUT_DIR = ROOT / "outputs" / "notebook_runs"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Canonical artifacts: {CANONICAL_OUTPUT_DIR}")
print(f"Notebook outputs: {NOTEBOOK_OUTPUT_DIR}")

In [ ]:
import json
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 150})

print("Results Dashboard")


def print_latex_table(df):
    try:
        print(f"\nLaTeX:\n{df.to_latex(index=False)}")
    except ImportError as exc:
        print(f"\nLaTeX export skipped: {exc}")

## Step 1: Load All Results

In [ ]:
results_dir = CANONICAL_OUTPUT_DIR
result_files = sorted(results_dir.glob("*_results.json"))

all_results = {}
for f in result_files:
    if f.name in {"adversarial_results.json"} or f.name.endswith("_manifest.json"):
        continue
    try:
        with open(f) as fp:
            data = json.load(fp)
        if not isinstance(data, dict):
            continue
        if not {"experiment", "task", "clean", "obfuscated"}.issubset(data):
            continue
        name = data.get("experiment", f.name)
        all_results[name] = data
        print(f"Loaded: {name}")
    except Exception as e:
        print(f"Failed to load {f}: {e}")

print(f"\nTotal: {len(all_results)} task-performance experiments loaded")

DASHBOARD_DIR = NOTEBOOK_OUTPUT_DIR / "dashboard"
FIGURES_DIR = DASHBOARD_DIR / "figures"
TABLES_DIR = DASHBOARD_DIR / "tables"

## Step 2: Classification Results

In [ ]:
cls_rows = []
for name, data in all_results.items():
    if data.get("task") == "classification":
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        cls_rows.append({
            "Experiment": name,
            "Model": data.get("model", ""),
            "Clean Acc": f"{clean.get('accuracy', 0):.4f}",
            "Obf Acc": f"{obf.get('accuracy', 0):.4f}",
            "Clean F1": f"{clean.get('f1', 0):.4f}",
            "Obf F1": f"{obf.get('f1', 0):.4f}",
        })

if cls_rows:
    df_cls = pd.DataFrame(cls_rows)
    print("Classification Results:")
    print(df_cls.to_string(index=False))
    print_latex_table(df_cls)
else:
    print("No classification results found")

## Step 3: Detection Results

In [ ]:
det_rows = []
for name, data in all_results.items():
    if data.get("task") in ("object_detection", "zero_shot_object_detection"):
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        det_rows.append({
            "Experiment": name,
            "Clean mAP": f"{clean.get('map', 0):.4f}",
            "Obf mAP": f"{obf.get('map', 0):.4f}",
            "Clean mAP@50": f"{clean.get('map_50', 0):.4f}",
            "Obf mAP@50": f"{obf.get('map_50', 0):.4f}",
        })

if det_rows:
    df_det = pd.DataFrame(det_rows)
    print("Detection Results:")
    print(df_det.to_string(index=False))
    print_latex_table(df_det)
else:
    print("No detection results found")

## Step 4: Segmentation Results

In [ ]:
med_rows = []
for name, data in all_results.items():
    if data.get("task") == "medical_segmentation":
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        med_rows.append({
            "Experiment": name,
            "Clean Dice": f"{clean.get('dice', 0):.4f}",
            "Obf Dice": f"{obf.get('dice', 0):.4f}",
            "Clean IoU": f"{clean.get('iou', 0):.4f}",
            "Obf IoU": f"{obf.get('iou', 0):.4f}",
        })

if med_rows:
    df_med = pd.DataFrame(med_rows)
    print("Medical Segmentation Results:")
    print(df_med.to_string(index=False))
    print_latex_table(df_med)
else:
    print("No medical segmentation results found")

## Step 5: Unified Results Overview

In [ ]:
METRIC_MAP = {
    "classification": ("accuracy", "Accuracy"),
    "zero_shot_classification": ("accuracy", "Accuracy"),
    "object_detection": ("map", "mAP"),
    "zero_shot_object_detection": ("map", "mAP"),
    "anomaly_detection": ("image_auroc", "Image AUROC"),
    "image_retrieval": ("recall_at_1", "Recall@1"),
    "image_text_retrieval": ("image_to_text_recall_at_1", "I2T R@1"),
    "medical_segmentation": ("dice", "Dice"),
    "image_captioning": ("bleu4", "BLEU-4"),
}

unified_rows = []
for name, data in all_results.items():
    task = data.get("task", "unknown")
    metric_key, metric_label = METRIC_MAP.get(task, ("accuracy", "Accuracy"))
    clean = data.get("clean", {})
    obf = data.get("obfuscated", {})
    clean_val = clean.get(metric_key, 0)
    obf_val = obf.get(metric_key, 0)
    delta = obf_val - clean_val
    unified_rows.append({
        "Experiment": name,
        "Task": task,
        "Metric": metric_label,
        "Clean": f"{clean_val:.4f}",
        "Obfuscated": f"{obf_val:.4f}",
        "Delta": f"{delta:+.4f}",
    })

if unified_rows:
    df_unified = pd.DataFrame(unified_rows)
    print("Unified Results Overview:")
    print(df_unified.to_string(index=False))
    print_latex_table(df_unified)
else:
    print("No results available")

## Step 6: Export Figures

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if all_results:
    names = list(all_results.keys())
    clean_vals = []
    obf_vals = []
    for name in names:
        data = all_results[name]
        task = data.get("task", "unknown")
        metric_key, _ = METRIC_MAP.get(task, ("accuracy", "Score"))
        clean = data.get("clean", {})
        obf = data.get("obfuscated", {})
        clean_vals.append(clean.get(metric_key, 0))
        obf_vals.append(obf.get(metric_key, 0))

    fig, ax = plt.subplots(figsize=(12, max(4, len(names) * 0.6)))
    y = range(len(names))
    height = 0.35
    ax.barh([i - height/2 for i in y], clean_vals, height, label="Clean", color="#2196F3")
    ax.barh([i + height/2 for i in y], obf_vals, height, label="Obfuscated", color="#FF9800")
    ax.set_yticks(list(y))
    ax.set_yticklabels(names)
    ax.set_xlabel("Task metric")
    ax.set_title("Clean vs. Obfuscated Performance")
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "performance_comparison.pdf", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "performance_comparison.png", bbox_inches="tight")
    plt.show()
    print(f"Figures saved to {FIGURES_DIR}")
else:
    print("No results to plot")

## Step 7: Export All Tables

In [ ]:
TABLES_DIR.mkdir(parents=True, exist_ok=True)

for name, df in [
    ("classification", df_cls if "df_cls" in dir() else None),
    ("detection", df_det if "df_det" in dir() else None),
    ("segmentation", df_med if "df_med" in dir() else None),
    ("unified", df_unified if "df_unified" in dir() else None),
]:
    if df is not None and not df.empty:
        path = TABLES_DIR / f"{name}_results.csv"
        df.to_csv(path, index=False)
        print(f"Saved {path}")

print("Export complete")